In [1]:
import re
import time
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datetime import datetime
from zoneinfo import ZoneInfo

In [2]:
BASE_URL = "https://www.inven.co.kr"
BOARD_URL = "https://www.inven.co.kr/board/maple/2300"
START_PAGE = 56
END_PAGE = 68
REQUEST_DELAY = 1.0

In [4]:

# 세션 설정
session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }
)

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET"},
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

# 게시글 저장용 리스트와 중복 방지용 세트
all_posts = []
visited_urls = set()

In [7]:
# 페이지별 수집
for page in range(START_PAGE, END_PAGE + 1):
    print("\n" + "=" * 60)
    print(f"{page}페이지 수집 시작")
    print("=" * 60)

    try:
        # 게시판 목록 페이지 요청
        response = session.get(
            BOARD_URL,
            params={"p": page},
            timeout=15
        )
        response.raise_for_status()

        if response.encoding is None or response.encoding.lower() == "iso-8859-1":
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 게시글 링크 추출
        articles = soup.select(".text-wrap .subject-link")
        print(f"[페이지 {page}] 게시글 링크 후보: {len(articles)}개")

        post_urls = []

        for article in articles:
            href = article.get("href")

            if not href:
                continue

            url = urljoin(BASE_URL, href)
            parsed = urlsplit(url)

            if not re.fullmatch(r"/board/maple/2300/\d+", parsed.path):
                continue

            clean_url = urlunsplit(
                (
                    parsed.scheme,
                    parsed.netloc,
                    parsed.path,
                    "",
                    ""
                )
            )

            post_urls.append(clean_url)

        post_urls = list(dict.fromkeys(post_urls))
        print(f"수집할 게시글: {len(post_urls)}개")

    except requests.RequestException as exc:
        print(f"[페이지 요청 실패] page={page}")
        print(exc)
        continue


    # 게시글 상세 내용 수집
    for index, url in enumerate(post_urls, start=1):
        if url in visited_urls:
            continue

        try:
            response = session.get(url, timeout=15)
            response.raise_for_status()

            if response.encoding is None or response.encoding.lower() == "iso-8859-1":
                response.encoding = response.apparent_encoding

            soup = BeautifulSoup(response.text, "html.parser")

            title_element = soup.select_one(".articleTitle")
            author_element = soup.select_one(".nickname")
            date_element = soup.select_one(".articleDate")
            content_element = soup.select_one(".contentBody")
            category_element = soup.select_one(".articleCategory")

            if title_element is None:
                raise ValueError(f"title을 찾지 못했습니다: {url}")

            if content_element is None:
                raise ValueError(f"content를 찾지 못했습니다: {url}")

            info_text = ""

            if date_element:
                node = date_element

                while node and getattr(node, "name", None) != "body":
                    text = node.get_text(" ", strip=True)

                    if "조회:" in text and "추천:" in text:
                        info_text = text
                        break

                    node = node.parent

            if category_element:
                category = (
                    category_element
                    .get_text(" ", strip=True)
                    .strip()
                    .strip("[]")
                    .strip()
                )
            else:
                category_match = re.search(r"\[([^\]]+)\]", info_text)

                category = (
                    category_match.group(1).strip()
                    if category_match
                    else None
                )

            # 4. views
            views_match = re.search(
                r"조회:\s*([\d,]+)",
                info_text
            )

            views = (
                int(views_match.group(1).replace(",", ""))
                if views_match
                else None
            )


            # 5. likes
            likes_match = re.search(
                r"추천:\s*([\d,]+)",
                info_text
            )

            likes = (
                int(likes_match.group(1).replace(",", ""))
                if likes_match
                else None
            )

            # 6. 결과 저장
            post = {
                "url": url,
                "category": category,
                "title": title_element.get_text(" ", strip=True),

                "author": (
                    author_element.get_text(" ", strip=True)
                    if author_element
                    else None
                ),

                "created_at": (
                    date_element.get_text(" ", strip=True)
                    if date_element
                    else None
                ),

                "views": views,
                "likes": likes,

                "content": content_element.get_text(
                    "\n",
                    strip=True
                ),

                "crawled_at": datetime.now(
                    ZoneInfo("Asia/Seoul")
                ).isoformat(timespec="seconds"),
            }

            all_posts.append(post)
            visited_urls.add(url)

            print(
                f"[{page}페이지 {index}/{len(post_urls)}] "
                f"{post['title']}"
            )

        except requests.RequestException as exc:
            print(f"[HTTP 오류] {url}")
            print(exc)

        except Exception as exc:
            print(f"[파싱 오류] {url}")
            print(exc)

        time.sleep(REQUEST_DELAY)

    time.sleep(REQUEST_DELAY)


56페이지 수집 시작
[페이지 56] 게시글 링크 후보: 50개
수집할 게시글: 50개
[56페이지 27/50] 가엔링 인트 36% 떴는데 이거 시세가 어떻게 되나요?
[56페이지 28/50] 레테 헥사스텟3 질문 있어요
[56페이지 29/50] 챌섭 메린이 메소로 어떤스펙업 먼저하나요??
[56페이지 30/50] 복귀유전데 뭐부터해야해용?
[56페이지 31/50] 소울인챈터 사라지나요?
[56페이지 32/50] 사냥할 때 컨티뉴어스 링 계속 끼면 돼요?
[56페이지 33/50] 챌썹 랩 276 썬콜 메이플 처음하는 메린이 메이린 최소컷 질문 ㅠ
[56페이지 34/50] 본섭 유기캐 본캐로 키우기.
[56페이지 35/50] 저장하지 않은 기본 성형,헤어 복구받을 수 있을까요?
[56페이지 36/50] 제네시스 무기 해방 관련 질문입니다!
[56페이지 37/50] [챌섭]혹시 무교 드메템 맞추면 원킬이 안뜨나요?
[56페이지 38/50] 메이플 브금 관련 질문
[56페이지 39/50] 헥사 초기화권 질문
[56페이지 40/50] 뉴비 현실적 목표 질문
[56페이지 41/50] 환산 직업마다 배율 다름?
[56페이지 42/50] 미트라 이정도면 비싼건가요?
[56페이지 43/50] 카레잠 어디에 바르는게 좋을까요?
[56페이지 44/50] 캐시 잘 못 수령했는데 ㅜㅜ
[56페이지 45/50] 뉴비 오늘 퇴근하고 챌섭해보려는데
[56페이지 46/50] 아니 자꾸 접속중인 아이디라고 팅기는데 왜이러지
[56페이지 47/50] 미트라 엠블 질문
[56페이지 48/50] 메린이 메획 + 주스텟 18 or 21퍼 어떰??
[56페이지 49/50] 오늘 검마 솔격으로 잡아야 되는건가요?
[56페이지 50/50] 메린이 레전 잠재 질문

57페이지 수집 시작
[페이지 57] 게시글 링크 후보: 50개
수집할 게시글: 50개
[57페이지 1/50] 메수라이브 잠재능력 설정 도와주세요
[57페이지 2/50] 오래된 캐릭터 정리된적 있나요?
[57페이지 3/50] 무소과금 서버는

In [8]:
columns = [
    "url",
    "category",
    "title",
    "author",
    "created_at",
    "views",
    "likes",
    "content",
    "crawled_at",
]

df = pd.DataFrame(
    all_posts,
    columns=columns
)

from pathlib import Path
print(Path.cwd())

root_dir = Path.cwd().parent.parent

output_path = "../../../data/raw/ksm_maple_inven_questions.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n총 수집 게시글: {len(df)}개")
print(f"CSV 파일 저장 완료: {output_path}")

df.head()

c:\Users\Playdata\Desktop\mle-01-p1-team3\src\data_collect\als00als

총 수집 게시글: 650개
CSV 파일 저장 완료: ../../../data/raw/ksm_maple_inven_questions.csv


,url,category,title,author,created_at,views,likes,content,crawled_at
0,https://www.inven.co.kr/board/maple/2300/358658,아이템,큐브 복구 가능할까요?,NaN,2026-07-02 05:02,685,0,제네무기 해방큐브 하다가 윗잠 3줄을 넘겨버렸습니다 3회씩 돌리는중이였는데 윗잠 3...,2026-08-16T15:22:32+09:00
1,https://www.inven.co.kr/board/maple/2300/358657,아이템,에메랄드 카레잠을 하트에 발랐는데 에디는 뭘 발라야 하나요,셜티,2026-07-02 03:58,820,0,챌 코인샵에서 레어 받아서 공 10 띄우면 되는지..\n아니면 에잠 사야되는지..,2026-08-16T15:22:33+09:00
2,https://www.inven.co.kr/board/maple/2300/358656,몬스터,돈 투자해서 하드 메이린 잡을만 한가요?,NaN,2026-07-02 03:49,1953,0,본섭 매주 주간보스로 70억정도 벌리는데\n챌섭캐 진심으로 키울 생각입니다.\n본섭...,2026-08-16T15:22:35+09:00
3,https://www.inven.co.kr/board/maple/2300/358655,몬스터,이지카링은 잡았는데 유챔세렌을 못잡아요,silenteye,2026-07-02 03:33,1017,0,듀블 렙285 헥사5.2입니다. 이지카링을 30초남기고 간신히깼는데 유챔세렌은 계속...,2026-08-16T15:22:36+09:00
4,https://www.inven.co.kr/board/maple/2300/358654,기타,카유잠 잘못샀는데 방법 없나요,레알밤판사,2026-07-02 03:28,609,0,딱히 쓸 템도 없는데 문의하면 1번만 복구해준다거나...,2026-08-16T15:22:37+09:00
